# Diagnosing two numerical problems in fracridge

This notebook demonstrates current behavior without patching or monkeypatching the installed package. It supplies local executable references for two numerical defects.

## Problem 1: cross-validation loses the intercept

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, KFold

from fracridge.fracridge import (
    FracRidgeRegressor,
    FracRidgeRegressorCV,
    fracridge,
)


In [ ]:
X_intercept = np.linspace(0.0, 1.0, 101)[:, np.newaxis]
y_intercept = 3.0 + 2.0 * X_intercept[:, 0]
cv = KFold(n_splits=5, shuffle=True, random_state=20260804)

current_cv = FracRidgeRegressorCV(
    fit_intercept=True,
    cv=cv,
    jit=False,
)
current_cv.fit(X_intercept, y_intercept, frac_grid=np.array([1.0]))
current_prediction = current_cv.predict(X_intercept)
current_intercept = float(np.asarray(current_cv.intercept_).squeeze())
current_max_error = float(np.max(np.abs(current_prediction - y_intercept)))
current_r2 = r2_score(y_intercept, current_prediction)

print(f"learned intercept: {current_intercept:.12g}")
print(f"maximum prediction error: {current_max_error:.12g}")
print(f"R-squared: {current_r2:.12g}")

assert abs(current_intercept - 3.0) > 2.5
assert current_max_error > 2.5


### Executable reference: let each estimator center raw data once

In [ ]:
def fit_cv_reference(X, y, *, frac_grid, cv):
    estimator = FracRidgeRegressor(fit_intercept=True, jit=False)
    search = GridSearchCV(estimator, {"fracs": np.asarray(frac_grid)}, cv=cv)
    return search.fit(X, y)


reference_cv = fit_cv_reference(
    X_intercept,
    y_intercept,
    frac_grid=np.array([1.0]),
    cv=cv,
)
reference_estimator = reference_cv.best_estimator_
reference_prediction = reference_estimator.predict(X_intercept)
reference_intercept = float(np.asarray(reference_estimator.intercept_).squeeze())
reference_max_error = float(
    np.max(np.abs(reference_prediction - y_intercept))
)
reference_r2 = r2_score(y_intercept, reference_prediction)

print(f"reference intercept: {reference_intercept:.12g}")
print(f"reference maximum error: {reference_max_error:.12g}")
print(f"reference R-squared: {reference_r2:.12g}")

assert np.isclose(reference_intercept, 3.0, atol=1e-10)
assert reference_max_error < 1e-10


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(X_intercept[:, 0], y_intercept, label="truth", linewidth=3)
ax.plot(
    X_intercept[:, 0],
    current_prediction,
    "--",
    label="current FracRidgeRegressorCV",
)
ax.plot(
    X_intercept[:, 0],
    reference_prediction,
    ":",
    label="raw-data GridSearchCV reference",
    linewidth=3,
)
ax.set(xlabel="X", ylabel="prediction", title="Intercept handling")
ax.legend()
fig.tight_layout()
plt.show()


### Production solution sketch for problem 1

`FracRidgeRegressorCV.fit` should validate raw inputs without centering, pass them directly to `GridSearchCV`, and copy the state of `best_estimator_` after refit. This sketch is not a package implementation; focused regression tests must be committed in RED state before either package fix.

## Problem 2: normal equations amplify numerical error

### Executable reference: decompose X directly

In [ ]:
def coefficient_norm_at_alpha(singular_values, ols_rotated, alpha):
    singular_values_sq = singular_values**2
    shrinkage = singular_values_sq / (singular_values_sq + alpha)
    return float(np.linalg.norm(shrinkage * ols_rotated))


def solve_fraction_alpha(singular_values, ols_rotated, fraction):
    if not 0.0 < fraction <= 1.0:
        raise ValueError("fraction must be in the interval (0, 1]")
    if fraction == 1.0:
        return 0.0

    target_norm = fraction * np.linalg.norm(ols_rotated)
    lower = 0.0
    upper = float(singular_values[0] ** 2)
    while coefficient_norm_at_alpha(singular_values, ols_rotated, upper) > target_norm:
        upper *= 2.0

    for _ in range(100):
        midpoint = (lower + upper) / 2.0
        midpoint_norm = coefficient_norm_at_alpha(
            singular_values, ols_rotated, midpoint
        )
        if midpoint_norm > target_norm:
            lower = midpoint
        else:
            upper = midpoint
    return (lower + upper) / 2.0


def direct_svd_components(X, y, tol):
    left, singular_values, right_transpose = np.linalg.svd(
        X, full_matrices=False
    )
    active = singular_values > tol
    if not np.any(active):
        raise ValueError("X has no singular values above tol")
    projected_y = left[:, active].T @ y
    ols_rotated = projected_y / singular_values[active]
    return singular_values[active], right_transpose[active], ols_rotated


def fracridge_direct_svd(X, y, fraction, *, tol=1e-10):
    singular_values, right_transpose, ols_rotated = direct_svd_components(
        X, y, tol
    )
    alpha = solve_fraction_alpha(singular_values, ols_rotated, fraction)
    singular_values_sq = singular_values**2
    shrinkage = singular_values_sq / (singular_values_sq + alpha)
    coefficients = right_transpose.T @ (shrinkage * ols_rotated)
    return coefficients, alpha


In [ ]:
def make_tall_design(base, perturbation, extras, epsilon):
    near_duplicate = base + epsilon * perturbation
    return np.column_stack((base, near_duplicate, extras))


rng = np.random.default_rng(20260804)
n_samples = 240
base_column = rng.normal(size=n_samples)
perturbation = rng.normal(size=n_samples)
extra_columns = rng.normal(size=(n_samples, 3))
true_coefficients = np.array([1.0, -1.0, 0.5, -0.25, 0.125])
epsilons = np.logspace(-4, -8, 5)
records = []

for epsilon in epsilons:
    design = make_tall_design(
        base_column, perturbation, extra_columns, epsilon
    )
    response = design @ true_coefficients
    least_squares = np.linalg.lstsq(design, response, rcond=None)[0]
    current_coefficients, _ = fracridge(
        design, response, fracs=1.0, jit=False
    )
    reference_coefficients, _ = fracridge_direct_svd(
        design, response, fraction=1.0
    )
    denominator = np.linalg.norm(least_squares)
    records.append(
        {
            "epsilon": epsilon,
            "condition": np.linalg.cond(design),
            "current_error": np.linalg.norm(
                current_coefficients - least_squares
            )
            / denominator,
            "reference_error": np.linalg.norm(
                reference_coefficients - least_squares
            )
            / denominator,
            "current_residual": np.linalg.norm(
                design @ current_coefficients - response
            ),
            "reference_residual": np.linalg.norm(
                design @ reference_coefficients - response
            ),
        }
    )

print(
    f"{'epsilon':>10} {'cond(X)':>14} {'current error':>16} "
    f"{'direct-SVD error':>18}"
)
for record in records:
    print(
        f"{record['epsilon']:10.1e} {record['condition']:14.6e} "
        f"{record['current_error']:16.6e} "
        f"{record['reference_error']:18.6e}"
    )

worst_case = records[-1]
assert worst_case["current_error"] > 0.1
assert worst_case["reference_error"] < 1e-8
assert worst_case["current_error"] > worst_case["reference_error"]


In [ ]:
fraction = 0.5
design = make_tall_design(
    base_column, perturbation, extra_columns, epsilons[-1]
)
response = design @ true_coefficients
ols_coefficients = np.linalg.lstsq(design, response, rcond=None)[0]
fractional_coefficients, fractional_alpha = fracridge_direct_svd(
    design, response, fraction
)
achieved_fraction = np.linalg.norm(fractional_coefficients) / np.linalg.norm(
    ols_coefficients
)

print(f"requested fraction: {fraction:.12g}")
print(f"achieved fraction: {achieved_fraction:.12g}")
print(f"reference alpha: {fractional_alpha:.12g}")

assert np.isclose(achieved_fraction, fraction, atol=1e-10)
assert fractional_alpha > 0.0


In [ ]:
conditions = np.array([record["condition"] for record in records])
current_errors = np.array([record["current_error"] for record in records])
reference_errors = np.array(
    [record["reference_error"] for record in records]
)
plot_floor = np.finfo(float).eps

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(conditions, current_errors, "o-", label="current X.T @ X path")
ax.loglog(
    conditions,
    np.maximum(reference_errors, plot_floor),
    "o-",
    label="direct economy-SVD reference",
)
ax.set(
    xlabel="condition number of X",
    ylabel="relative coefficient error",
    title="Normal equations square the conditioning problem",
)
ax.legend()
fig.tight_layout()
plt.show()


### Production solution sketch for problem 2

`_do_svd` should always compute an economy SVD of `X` directly, apply the singular-value tolerance before division, and preserve vector, multi-target, tall, and wide shapes. This sketch is not a package implementation; focused regression tests must be committed in RED state before either package fix.

## Test-first repair sequence

1. Add focused function-based regression tests for intercept preservation and ill-conditioned tall-matrix accuracy.
2. Run each focused test and observe the intended numerical assertion fail.
3. Commit those failing tests without implementation changes.
4. Implement one production fix at a time, running the focused test after each change.
5. Run the complete suite before refactoring, then refactor only while green.